# Freeway — Live CNN/DQN Training + Data Collection
Collects experience AND trains a CNN-based DQN model at the same time.

In [5]:
!pip install gymnasium[atari] ale-py opencv-python torch torchvision

## Step 1 — Imports

In [6]:
import ale_py
import gymnasium as gym
import numpy as np
import cv2
import random
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Step 2 — Frame Processing (same as before)

In [7]:
def preprocess_frame(frame):
    """Grayscale + resize to 84x84 + normalize to [0,1]."""
    grayscale = np.mean(frame, axis=2).astype(np.float32)
    resized   = cv2.resize(grayscale, (84, 84))
    return resized / 255.0

def stack_frames(frame_stack, new_frame):
    """Drop oldest frame, append newest (FIFO queue of 4)."""
    frame_stack.append(new_frame)
    if len(frame_stack) > 4:
        frame_stack.pop(0)
    return frame_stack

## Step 3 — CNN Model (DQN Architecture)

This is the same architecture used in the original DeepMind DQN paper:
- 3 Convolutional layers to extract visual features from the frames
- 2 Fully Connected layers to decide which action to take
- Output = 3 values (one score per action: NOOP, UP, DOWN)

In [8]:
class DQN(nn.Module):
    def __init__(self, n_actions=3):
        super(DQN, self).__init__()

        # CNN layers — extract features from (4, 84, 84) stacked frames
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),   # → (32, 20, 20)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),  # → (64, 9, 9)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),  # → (64, 7, 7)
            nn.ReLU()
        )

        # Fully connected layers — decide the action
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)   # 3 outputs: NOOP, UP, DOWN
        )

    def forward(self, x):
        x = self.conv(x)                # Extract visual features
        x = x.view(x.size(0), -1)      # Flatten
        return self.fc(x)              # Return Q-values for each action


# Create the model and print its structure
model = DQN(n_actions=3).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {total_params:,}")

DQN(
  (conv): Sequential(
    (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
  )
  (fc): Sequential(
    (0): Linear(in_features=3136, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=3, bias=True)
  )
)

Total trainable parameters: 1,685,667


## Step 4 — Replay Buffer (stores collected data)

In [9]:
class ReplayBuffer:
    """Stores experience tuples: (state, action, reward, next_state, done)."""
    def __init__(self, capacity=50_000):
        self.buffer = deque(maxlen=capacity)   # auto-drops old data when full

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states),      dtype=torch.float32).to(device),
            torch.tensor(actions,               dtype=torch.long).to(device),
            torch.tensor(rewards,               dtype=torch.float32).to(device),
            torch.tensor(np.array(next_states), dtype=torch.float32).to(device),
            torch.tensor(dones,                 dtype=torch.float32).to(device),
        )

    def __len__(self):
        return len(self.buffer)

replay_buffer = ReplayBuffer(capacity=50_000)
print("Replay buffer ready (capacity: 50,000 experiences)")

Replay buffer ready (capacity: 50,000 experiences)


## Step 5 — Training Settings

In [10]:
# Hyperparameters
TOTAL_STEPS    = 100_000   # total steps to collect and train on
BATCH_SIZE     = 32        # how many experiences to learn from at once
GAMMA          = 0.99      # how much to value future rewards (0=now, 1=future)
LR             = 1e-4      # learning rate
LEARN_START    = 10_000    # start training only after this many steps collected
LEARN_EVERY    = 4         # train the model every 4 steps
EPSILON_START  = 1.0       # start fully random
EPSILON_END    = 0.1       # end at 10% random
EPSILON_DECAY  = 100_000   # how many steps to decay epsilon over

optimizer = optim.Adam(model.parameters(), lr=LR)
loss_fn   = nn.MSELoss()

print("Settings ready!")
print(f"  Will collect {TOTAL_STEPS:,} steps")
print(f"  Training starts after {LEARN_START:,} steps")
print(f"  Epsilon: {EPSILON_START} → {EPSILON_END} over {EPSILON_DECAY:,} steps")

Settings ready!
  Will collect 100,000 steps
  Training starts after 10,000 steps
  Epsilon: 1.0 → 0.1 over 100,000 steps


## Step 6 — Live Training Loop
Collects data AND trains the CNN at the same time.

In [ ]:
# ── Environment setup ────────────────────────────────────────────
gym.register_envs(ale_py)

# We need TWO environments:
# 1. rgb — for the CNN frames (visual input)
# 2. ram — for reading the chicken's exact position
env_rgb = gym.make("ALE/Freeway-v5", render_mode="rgb_array", obs_type="rgb")
env_ram = gym.make("ALE/Freeway-v5", render_mode="rgb_array", obs_type="ram")

obs_rgb, _ = env_rgb.reset(seed=42)
obs_ram, _ = env_ram.reset(seed=42)

first_frame = preprocess_frame(obs_rgb)
frame_stack = [first_frame] * 4

# ── Reward shaping function ──────────────────────────────────────
def shape_reward(obs_ram, action, base_reward, prev_y, done):
    """
    Custom reward system:
      +0.05  each step the chicken moves UP (making progress)
      -0.05  each step the chicken moves DOWN (going backwards)
      -0.3   when the chicken gets hit by a car (position resets)
      +10.0  when the chicken fully crosses (base_reward == 1)
    """
    shaped = base_reward  # start with the game's original reward

    # Read chicken Y position from RAM (byte 14)
    # Lower number = higher on screen = closer to crossing
    chicken_y = int(obs_ram[14])

    # Big reward for fully crossing
    if base_reward == 1:
        shaped += 10.0

    # Small reward for moving up (making progress toward crossing)
    if chicken_y < prev_y:
        shaped += 0.05

    # Small penalty for moving down (going backwards)
    if chicken_y > prev_y:
        shaped -= 0.05

    # Penalty for getting hit — detected when position suddenly jumps back
    # A hit sends the chicken back more than 5 pixels at once
    if chicken_y - prev_y > 5:
        shaped -= 0.3

    return shaped, chicken_y


# ── Tracking metrics ─────────────────────────────────────────────
episode_rewards = []
episode_reward  = 0
losses          = []
episode         = 0
prev_y          = int(obs_ram[14])   # starting chicken Y position

print("Starting live data collection + CNN training with reward shaping...")
print()
print("Reward system:")
print("  +10.0  chicken fully crosses the road")
print("  +0.05  chicken moves UP (progress)")
print("  -0.05  chicken moves DOWN (going back)")
print("  -0.30  chicken gets hit by a car")
print()

# ── Main training loop ───────────────────────────────────────────
for step in range(TOTAL_STEPS):

    # --- Epsilon-greedy action selection ---
    epsilon = EPSILON_END + (EPSILON_START - EPSILON_END) * \
              max(0, (EPSILON_DECAY - step) / EPSILON_DECAY)

    current_state = np.stack(frame_stack, axis=0)   # (4, 84, 84)

    if random.random() < epsilon:
        action = env_rgb.action_space.sample()       # random
    else:
        state_t = torch.tensor(current_state, dtype=torch.float32)\
                       .unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = model(state_t)
        action = q_values.argmax().item()            # CNN picks action

    # --- Step BOTH environments with the same action ---
    obs_rgb, base_reward, terminated, truncated, _ = env_rgb.step(action)
    obs_ram, _,           _,          _,          _ = env_ram.step(action)
    done = terminated or truncated

    # --- Apply shaped reward ---
    shaped_reward, prev_y = shape_reward(
        obs_ram, action, base_reward, prev_y, done
    )
    episode_reward += shaped_reward

    # --- Build next stacked state ---
    gray_frame  = preprocess_frame(obs_rgb)
    frame_stack = stack_frames(frame_stack, gray_frame)
    next_state  = np.stack(frame_stack, axis=0)     # (4, 84, 84)

    # --- Store shaped experience in replay buffer ---
    replay_buffer.push(current_state, action, shaped_reward, next_state, done)

    # --- Train CNN ---
    if len(replay_buffer) >= LEARN_START and step % LEARN_EVERY == 0:
        states_b, actions_b, rewards_b, next_states_b, dones_b = \
            replay_buffer.sample(BATCH_SIZE)

        q_values_b = model(states_b).gather(1, actions_b.unsqueeze(1)).squeeze()

        with torch.no_grad():
            max_next_q = model(next_states_b).max(1)[0]
            targets    = rewards_b + GAMMA * max_next_q * (1 - dones_b)

        loss = loss_fn(q_values_b, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    # --- Reset on episode end ---
    if done:
        episode        += 1
        episode_rewards.append(episode_reward)
        episode_reward  = 0

        obs_rgb, _ = env_rgb.reset()
        obs_ram, _ = env_ram.reset()
        frame_stack = [preprocess_frame(obs_rgb)] * 4
        prev_y      = int(obs_ram[14])   # reset chicken position tracker

    # --- Progress update every 10,000 steps ---
    if (step + 1) % 10_000 == 0:
        avg_reward = np.mean(episode_rewards[-10:]) if episode_rewards else 0
        avg_loss   = np.mean(losses[-100:])         if losses         else 0
        print(f"Step {step+1:>7,} | Episodes: {episode:>4} | "
              f"Avg Reward: {avg_reward:.2f} | "
              f"Avg Loss: {avg_loss:.4f} | "
              f"Epsilon: {epsilon:.3f}")

env_rgb.close()
env_ram.close()
print("\nTraining complete!")

Starting live data collection + CNN training with reward shaping...

Reward system:
  +10.0  chicken fully crosses the road
  +0.05  chicken moves UP (progress)
  -0.05  chicken moves DOWN (going back)
  -0.30  chicken gets hit by a car



## Step 7 — Plot Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Reward per episode
ax1.plot(episode_rewards, alpha=0.4, color='steelblue', label='Episode Reward')
if len(episode_rewards) >= 10:
    moving_avg = np.convolve(episode_rewards, np.ones(10)/10, mode='valid')
    ax1.plot(moving_avg, color='navy', linewidth=2, label='10-ep Moving Avg')
ax1.set_title('Reward per Episode')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Training loss
ax2.plot(losses, alpha=0.3, color='tomato', label='Loss')
if len(losses) >= 100:
    moving_loss = np.convolve(losses, np.ones(100)/100, mode='valid')
    ax2.plot(moving_loss, color='darkred', linewidth=2, label='100-step Moving Avg')
ax2.set_title('CNN Training Loss')
ax2.set_xlabel('Training Step')
ax2.set_ylabel('MSE Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Freeway DQN Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_results.png', dpi=150)
plt.show()
print("Plot saved to training_results.png")

## Step 8 — Save the Trained Model + Data

In [ ]:
# Save the CNN model weights
torch.save(model.state_dict(), "freeway_dqn_model.pth")
print("CNN model saved to freeway_dqn_model.pth")

# Save the collected experience data
buffer_data = list(replay_buffer.buffer)
states_arr      = np.array([e[0] for e in buffer_data], dtype=np.float32)
actions_arr     = np.array([e[1] for e in buffer_data], dtype=np.int32)
rewards_arr     = np.array([e[2] for e in buffer_data], dtype=np.float32)
next_states_arr = np.array([e[3] for e in buffer_data], dtype=np.float32)
dones_arr       = np.array([e[4] for e in buffer_data], dtype=bool)

np.savez_compressed(
    "freeway_data_100k.npz",
    states=states_arr,
    actions=actions_arr,
    rewards=rewards_arr,
    next_states=next_states_arr,
    dones=dones_arr
)
print("Experience data saved to freeway_data_100k.npz")

# Final summary
print(f"\n--- Final Summary ---")
print(f"Total steps      : {TOTAL_STEPS:,}")
print(f"Episodes played  : {episode}")
print(f"Best reward      : {max(episode_rewards) if episode_rewards else 0}")
print(f"Avg reward       : {np.mean(episode_rewards):.2f}" if episode_rewards else "N/A")
print(f"Training updates : {len(losses):,}")
print(f"Final epsilon    : {EPSILON_END}")